Best Selling Item 
Find the best-selling item for each month. The best-selling item is determined by the highest total sales amount, calculated as: total_paid = unitprice * quantity. A negative quantity indicates a return or cancellation (the invoice number begins with 'C'. To calculate sales, ignore returns and cancellations. Output the month, description of the item, and the total amount paid. Table online_retail

table: online_retail
columns
invoice_number: if it starts with C its a cancellation
desc
qty
up
invoice_date



In [0]:
data = [
    ("10001", "LUNCH BAG SPACEBOY DESIGN", 10, 7.426, "2023-01-10"),
    ("10002", "REGENCY CAKESTAND 3 TIER", 5, 7.65, "2023-02-15"),
    ("10003", "PAPER BUNTING WHITE LACE", 20, 5.1, "2023-03-12"),
    ("10004", "SPACEBOY LUNCH BOX", 6, 3.9, "2023-04-18"),
    ("10005", "PAPER BUNTING WHITE LACE", 10, 5.1, "2023-05-20"),
    ("C10006", "LUNCH BAG SPACEBOY DESIGN", -5, 7.426, "2023-01-25"),
    ("10007", "REGENCY CAKESTAND 3 TIER", -2, 7.65, "2023-02-28")
]
columns = ["invoice_no", "description", "quantity", "unit_price", "invoice_date"]

df = spark.createDataFrame(data, columns)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df_filtered = df.filter(
    (~col("invoice_no").startswith("C")) &  # ✅ invoice_no
    (col("quantity") > 0)
)

In [0]:
#Total_paid column added
df_with_total = df_filtered.withColumn(
    "total_paid",
    col("quantity") * col("unit_price")
)



In [0]:
df_with_total.display()

invoice_no,description,quantity,unit_price,invoice_date,total_paid
10001,LUNCH BAG SPACEBOY DESIGN,10,7.426,2023-01-10,74.26
10002,REGENCY CAKESTAND 3 TIER,5,7.65,2023-02-15,38.25
10003,PAPER BUNTING WHITE LACE,20,5.1,2023-03-12,102.0
10004,SPACEBOY LUNCH BOX,6,3.9,2023-04-18,23.4
10005,PAPER BUNTING WHITE LACE,10,5.1,2023-05-20,51.0


In [0]:
#convert invoice_date to months only
df_with_month = df_with_total.withColumn(
    "month",
    month(to_date(col("invoice_date")))
)


In [0]:
df_with_month.display()

invoice_no,description,quantity,unit_price,invoice_date,total_paid,month
10001,LUNCH BAG SPACEBOY DESIGN,10,7.426,2023-01-10,74.26,1
10002,REGENCY CAKESTAND 3 TIER,5,7.65,2023-02-15,38.25,2
10003,PAPER BUNTING WHITE LACE,20,5.1,2023-03-12,102.0,3
10004,SPACEBOY LUNCH BOX,6,3.9,2023-04-18,23.4,4
10005,PAPER BUNTING WHITE LACE,10,5.1,2023-05-20,51.0,5


In [0]:
#Agg per item per month

df_grouped= df_with_month.groupBy("month","description") \
    .sum("total_paid")\
        .withColumnRenamed("sum(total_paid)","total_paid")


In [0]:
df_grouped.display()

month,description,total_paid
1,LUNCH BAG SPACEBOY DESIGN,74.26
2,REGENCY CAKESTAND 3 TIER,38.25
3,PAPER BUNTING WHITE LACE,102.0
4,SPACEBOY LUNCH BOX,23.4
5,PAPER BUNTING WHITE LACE,51.0


In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("month").orderBy("total_paid")

df_ranked = df_grouped.withColumn("rank", row_number().over(window_spec))
df_ranked.display()



month,description,total_paid,rank
1,LUNCH BAG SPACEBOY DESIGN,74.26,1
2,REGENCY CAKESTAND 3 TIER,38.25,1
3,PAPER BUNTING WHITE LACE,102.0,1
4,SPACEBOY LUNCH BOX,23.4,1
5,PAPER BUNTING WHITE LACE,51.0,1


In [0]:
df_result = df_ranked.filter(col("rank")==1)\
    .select("month","description","total_paid")
df_result.display()

month,description,total_paid
1,LUNCH BAG SPACEBOY DESIGN,74.26
2,REGENCY CAKESTAND 3 TIER,38.25
3,PAPER BUNTING WHITE LACE,102.0
4,SPACEBOY LUNCH BOX,23.4
5,PAPER BUNTING WHITE LACE,51.0
